# Demo 12b — Convolution, pooling and receptive fields

**Class 12 · C1–C2** (slides: `slides/12_neural_networks.md`)

* **C1:** a convolution by hand on a 1-D signal and on a digit; three hand-made kernels; the output-size formula against Keras.
* **C2:** equivariance to shifts, the gradient of a kernel against `jax.grad`, the receptive field of stacked layers, pooling and the stride failure case.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import jax
import jax.numpy as jnp
import keras
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from faa import demo, neural, plotlib
from faa.demo import show, slide

demo.setup("12")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0

## Slide: A convolution by hand (C1)

Signal $x = (1, 2, 0, -1, 3)$, kernel $w = (1, 0, -1)$ (a difference of the two neighbours), no padding, stride 1:
$y_i = \sum_j w_j\, x_{i+j}$. Deep learning calls this a convolution; mathematically it is the cross-correlation.

In [ ]:
x = np.array([1.0, 2, 0, -1, 3])
w = np.array([1.0, 0, -1])
y = np.array([sum(w[j] * x[i + j] for j in range(3)) for i in range(len(x) - 2)])
slide("A convolution by hand")
print("x =", x, "w =", w)
for i, v in enumerate(y):
    terms = " + ".join(f"({w[j]:+.0f})({x[i + j]:+.0f})" for j in range(3))
    print(f"y_{i + 1} = {terms} = {v:+.0f}")
print(
    "np.correlate agrees:",
    np.allclose(y, np.correlate(x, w, "valid")),
    "; kernel flipped, np.convolve:",
    np.allclose(y, np.convolve(x, w[::-1], "valid")),
)

## Slide: Three kernels on a digit (C1)

In [ ]:
X, lab = neural.load_digits_images()
img = X[np.where(lab == 7)[0][0]]
kernels = {
    "vertical edge": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], float),
    "horizontal edge": np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], float),
    "blur": np.ones((3, 3)) / 9,
}
maps = {k: neural.conv2d(img, v[:, :, None, None])[..., 0] for k, v in kernels.items()}
f, ax = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.2))
ax[0].imshow(img[..., 0], cmap="gray_r")
ax[0].set_title("digit 7 (8x8)")
for a, (k, m) in zip(ax[1:], maps.items()):
    a.imshow(m, cmap="RdBu_r", vmin=-np.abs(m).max(), vmax=np.abs(m).max())
    a.set_title(k)
for a in ax:
    a.axis("off")
plotlib.save(f, "c1_kernels")
slide("Three kernels on a digit")
for k, m in maps.items():
    print(f"{k:16s} output {m.shape}, min {m.min():+.2f}, max {m.max():+.2f}")

## Slide: Output size (C1)

$n_{out} = \lfloor (n + 2p - k)/s \rfloor + 1$ for input length $n$, kernel $k$, padding $p$, stride $s$. Checked against Keras.

In [ ]:
rows = []
for n, k, p, s in [
    (8, 3, 0, 1),
    (8, 3, 1, 1),
    (8, 3, 0, 2),
    (8, 5, 2, 1),
    (28, 5, 0, 1),
    (28, 3, 1, 2),
    (224, 7, 3, 2),
]:
    layer = keras.layers.Conv2D(1, k, strides=s, padding="valid")
    pad = keras.layers.ZeroPadding2D(p) if p else keras.layers.Identity()
    out = layer(pad(np.zeros((1, n, n, 1), "float32")))
    rows.append({"n": n, "k": k, "p": p, "s": s, "formula": neural.out_size(n, k, p, s), "keras": out.shape[1]})
slide("Output size of a convolution")
show(pl.DataFrame(rows))
assert all(r["formula"] == r["keras"] for r in rows)
f, ax = plotlib.fig("col", 1.6)
Ls = np.arange(4, 33)
for (k, p, s), c in zip([(3, 0, 1), (3, 1, 1), (3, 0, 2), (3, 1, 2)], (BLUE, GREEN, ORANGE, RED)):
    ax.plot(Ls, [neural.out_size(int(n), k, p, s) for n in Ls], color=c, label=f"$P={p}$, $S={s}$")
ax.set_xlabel("input length $L$ ($F = 3$)")
ax.set_ylabel("output length $L_{out}$")
ax.legend(frameon=False, fontsize=6)
plotlib.save(f, "c1_out_size")

## Slide: Same weights everywhere = equivariance (C2)

Move the digit one pixel right and convolve: the feature map is the old map moved one pixel right (away from the border).

In [ ]:
shifted = neural.shift_images(img[None], 1, 0)[0]
k3 = kernels["vertical edge"][:, :, None, None]
a = neural.conv2d(shifted, k3, pad=1)[..., 0]
b = neural.shift_images(neural.conv2d(img, k3, pad=1)[None], 1, 0)[0, ..., 0]
slide("Equivariance to a shift")
print(f"max |conv(shift x) - shift(conv x)| inside the image: {np.abs(a[:, 2:-1] - b[:, 2:-1]).max():.1e}")

## Slide: The gradient of a kernel (C2)

For $y = w \star x$ and a loss $L$, $\partial L/\partial w_j = \sum_i (\partial L/\partial y_i)\, x_{i+j}$: the kernel gradient is a correlation of the input with the upstream gradient. Checked with `jax.grad`.

In [ ]:
rng = np.random.default_rng(SEED)
x = rng.normal(size=12)
w = rng.normal(size=3)
t = rng.normal(size=10)


def loss(w):
    y = jnp.array([jnp.dot(w, x[i : i + 3]) for i in range(10)])
    return 0.5 * jnp.sum((y - t) ** 2)


y = np.array([w @ x[i : i + 3] for i in range(10)])
g_hand = np.array([sum((y[i] - t[i]) * x[i + j] for i in range(10)) for j in range(3)])
g_jax = np.array(jax.grad(loss)(jnp.array(w)))
slide("The gradient of a kernel")
print("by hand :", np.round(g_hand, 5))
print("jax.grad:", np.round(g_jax, 5))
print("params: kernel 3 vs dense 12 x 10 =", 120)

## Slide: Receptive field (C2)

One unit of layer $L$ sees $r_L = 1 + \sum_l (k_l - 1)\prod_{m<l} s_m$ input pixels per side.

In [ ]:
def receptive(layers):
    r, jump = 1, 1
    for k, s in layers:
        r += (k - 1) * jump
        jump *= s
    return r


archs = {
    "3x3 x1": [(3, 1)],
    "3x3 x2": [(3, 1)] * 2,
    "3x3 x3": [(3, 1)] * 3,
    "5x5 x1": [(5, 1)],
    "3x3, pool 2, 3x3": [(3, 1), (2, 2), (3, 1)],
    "3x3, pool 2, 3x3, pool 2, 3x3": [(3, 1), (2, 2), (3, 1), (2, 2), (3, 1)],
}
rows = [
    {"layers": k, "receptive field": receptive(v), "kernel weights (1 channel)": sum(a * a for a, _ in v)}
    for k, v in archs.items()
]
slide("Receptive field")
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.5)
depth = np.arange(1, 8)
ax.plot(depth, [receptive([(3, 1)] * d) for d in depth], "o-", color=BLUE, label="3x3 stack")
ax.plot(depth, [receptive([(3, 1), (2, 2)] * d) for d in depth], "s-", color=ORANGE, label="3x3 + pool 2")
ax.set_xlabel("blocks")
ax.set_ylabel("receptive field (pixels)")
ax.legend(frameon=False)
plotlib.save(f, "c2_receptive")

## Slide: Pooling (C2)

Max pooling keeps the strongest response in each 2x2 block. A one-pixel shift moves the pooled map less than it moves the feature map.

In [ ]:
fm = neural.conv2d(img, k3, pad=1)
fm_s = neural.conv2d(shifted, k3, pad=1)
p0, p1 = neural.max_pool(np.abs(fm)), neural.max_pool(np.abs(fm_s))


def rel(a, b):
    return np.linalg.norm(a - b) / np.linalg.norm(a)


slide("Pooling")
print(f"feature map {fm.shape}: relative change after a 1-pixel shift {rel(np.abs(fm), np.abs(fm_s)):.3f}")
print(f"pooled map  {p0.shape}: relative change after a 1-pixel shift {rel(p0, p1):.3f}")

## Failure case: stride breaks equivariance

With stride 2 only every other position is computed: a shift of 2 pixels gives an exact shift of the output, a shift of 1 does not.

In [ ]:
sig = np.zeros(16)
sig[5:8] = [1, 3, 1]
kk = np.array([1.0, 1, 1])
rows = []
for s in (1, 2):
    out = np.correlate(sig, kk, "valid")[::s]
    for d in (1, 2):
        moved = np.correlate(np.roll(sig, d), kk, "valid")[::s]
        err = min(float(np.abs(moved - np.roll(out, r)).max()) for r in range(3))
        rows.append({"stride": s, "input shift d": d, "mismatch": err})
slide("Stride and equivariance")
show(pl.DataFrame(rows))
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.4))
ax[0].stem(np.arange(16), sig, linefmt=BLUE, markerfmt="o", basefmt=" ", label="input")
ax[0].set_xlabel("position")
ax[0].set_title("input (a bump of three values)")
out0 = np.correlate(sig, kk, "valid")
for d, c in zip((0, 1, 2), (BLUE, ORANGE, RED)):
    o = np.correlate(np.roll(sig, d), kk, "valid")[::2]
    ax[1].plot(np.arange(len(o)), o, "o-", color=c, label=f"input shifted by {d}")
ax[1].set_xlabel("output index (stride 2)")
ax[1].set_title("output")
ax[1].set_ylim(0, 7.5)
ax[1].legend(frameon=False, fontsize=5.5, loc="upper right")
plotlib.save(f, "c2_stride")